# TabTransformer — Búsqueda de Hiperparámetros con Optuna (L1, L2, L3)

**Framework:** pytorch-tabular (`TabTransformerConfig`) — gestiona embeddings, transformer y MLP de salida automáticamente.

**Features:** 8 continuas + 3 categóricas (acuity, chiefcomplaint top-200, arrival_transport).

**Estrategia:** Optuna `TPESampler(seed=42)`, **30 trials × 3 targets**, maximizando AUROC val. Reentrenamiento final: max_epochs=30, patience=7.

## 1. Setup

Importaciones, configuración de semillas de reproducibilidad y constantes globales. `N_TRIALS=30` equipara el presupuesto de búsqueda del TabTransformer con el resto de arquitecturas del estudio.

In [16]:
import os, json, pickle, time, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import torch
from pytorch_tabular import TabularModel
from pytorch_tabular.models import TabTransformerConfig
from pytorch_tabular.config import DataConfig, TrainerConfig, OptimizerConfig
from dotenv import load_dotenv
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

load_dotenv(dotenv_path=Path('../../.env'), override=True)
if not os.getenv('MIMIC_IV_ED_PATH'):
    load_dotenv(dotenv_path=Path('.env'), override=True)

DATA          = Path(os.getenv('MIMIC_IV_ED_PATH', ''))
PROCESSED_DIR = Path('../../data/processed')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

N_TRIALS     = 30
TARGETS      = ['L1', 'L2', 'L3']
TARGET_NAMES = {'L1': 'Ingreso hospitalario', 'L2': 'Resultado crítico', 'L3': 'Intervención crítica'}
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 150, 'font.size': 10})

Device: cuda


## 2. Carga de datos + feature engineering

pytorch-tabular acepta DataFrames directamente y gestiona los embeddings de categóricas internamente. Solo se preprocesa `chiefcomplaint` para limitar el vocabulario a los top-200 valores más frecuentes (evitar explosión de embeddings). Las continuas se imputan con la mediana de train.

In [17]:
df_train = pd.read_parquet(PROCESSED_DIR / 'train.parquet')
df_val   = pd.read_parquet(PROCESSED_DIR / 'val.parquet')

# n_medications desde medrecon
df_medrecon = pd.read_csv(DATA / 'medrecon.csv', low_memory=False, usecols=['stay_id'])
n_meds = df_medrecon.groupby('stay_id').size().rename('n_medications').reset_index()
for df in [df_train, df_val]:
    df.drop(columns=['n_medications'], errors='ignore', inplace=True)
df_train = df_train.merge(n_meds, on='stay_id', how='left')
df_val   = df_val.merge(n_meds, on='stay_id', how='left')
df_train['n_medications'] = df_train['n_medications'].fillna(0).astype(int)
df_val['n_medications']   = df_val['n_medications'].fillna(0).astype(int)

# pain almacenado como str en los parquets
df_train['pain'] = pd.to_numeric(df_train['pain'], errors='coerce')
df_val['pain']   = pd.to_numeric(df_val['pain'],   errors='coerce')

# --- Categóricas ---
# chiefcomplaint → top-200 + '__other__'
CHIEFCOMPLAINT_TOP_N = 200
cc_counts = df_train['chiefcomplaint'].fillna('').str.lower().str.strip().value_counts()
top_cc = set(cc_counts.head(CHIEFCOMPLAINT_TOP_N).index)

def map_cc(series):
    s = series.fillna('').str.lower().str.strip()
    return s.where(s.isin(top_cc), '__other__')

for df in [df_train, df_val]:
    df['cc_top200']              = map_cc(df['chiefcomplaint'])
    df['acuity_str']             = df['acuity'].fillna(0).astype(int).clip(0, 5).astype(str)
    df['arrival_transport_str']  = df['arrival_transport'].fillna('unknown').str.lower().str.strip()

# --- Continuas: imputación con mediana de train ---
CONT_FEATURES = ['temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'n_medications', 'pain']
CAT_FEATURES  = ['acuity_str', 'cc_top200', 'arrival_transport_str']

imputer = SimpleImputer(strategy='median')
df_train[CONT_FEATURES] = imputer.fit_transform(df_train[CONT_FEATURES])
df_val[CONT_FEATURES]   = imputer.transform(df_val[CONT_FEATURES])

print(f'Train: {df_train.shape} | Val: {df_val.shape}')
print(f'Continuas: {CONT_FEATURES}')
print(f'Categóricas: {CAT_FEATURES}')
print(f'cc_top200 unique (train): {df_train["cc_top200"].nunique()}')
for t in TARGETS:
    print(f'  {t} prevalencia train={df_train[t].mean()*100:.2f}%  val={df_val[t].mean()*100:.2f}%')

Train: (278320, 25) | Val: (59640, 25)
Continuas: ['temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'n_medications', 'pain']
Categóricas: ['acuity_str', 'cc_top200', 'arrival_transport_str']
cc_top200 unique (train): 201
  L1 prevalencia train=38.55%  val=38.31%
  L2 prevalencia train=1.54%  val=1.47%
  L3 prevalencia train=0.56%  val=0.61%


## 3. Función objetivo Optuna con pytorch-tabular

`TabularModel` encapsula la arquitectura TabTransformer completa. Cada trial crea un modelo nuevo, lo entrena con `model.fit()` y lo descarta. No se necesita implementar el bucle de entrenamiento manual.

In [22]:
def build_tabular_model(embed_dim, n_heads, n_blocks, dropout, out_ff_layers,
                        lr, wd, batch_size, target, max_epochs, patience,
                        checkpoints=None, load_best=False):
    data_config = DataConfig(
        target=[target],
        continuous_cols=CONT_FEATURES,
        categorical_cols=CAT_FEATURES,
        num_workers=0,
    )
    model_config = TabTransformerConfig(
        task='classification',
        input_embed_dim=embed_dim,
        num_heads=n_heads,
        num_attn_blocks=n_blocks,
        attn_dropout=dropout,
        ff_dropout=dropout,
        head_config={'layers': out_ff_layers, 'dropout': dropout},
        learning_rate=lr,
    )
    trainer_config = TrainerConfig(
        max_epochs=max_epochs,
        batch_size=batch_size,
        early_stopping='valid_loss',
        early_stopping_patience=patience,
        checkpoints=checkpoints,
        load_best=load_best,
        progress_bar='none',
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        trainer_kwargs={'enable_model_summary': False},
    )
    optimizer_config = OptimizerConfig(
        optimizer='Adam',
        optimizer_params={'weight_decay': wd},
    )
    return TabularModel(
        data_config=data_config,
        model_config=model_config,
        optimizer_config=optimizer_config,
        trainer_config=trainer_config,
    )


def get_prob_col(pred_df: pd.DataFrame) -> str:
    """Columna de P(clase=1), compatible con pytorch-tabular >=1.0 y <1.0."""
    col = next((c for c in pred_df.columns if c.endswith('_1_probability')), None)
    if col is None:
        col = next((c for c in pred_df.columns if c.endswith('_1')), None)
    if col is None:
        raise ValueError(f"No probability col found. Got: {list(pred_df.columns)}")
    return col


def get_auroc(model, val_df, target):
    pred = model.predict(val_df)
    return roc_auc_score(val_df[target].values, pred[get_prob_col(pred)].values)

## 4. Espacio de búsqueda Optuna

| Parámetro | Rango |
|-----------|-------|
| `embed_dim` | {16, 32, 64} |
| `n_heads` | {2, 4, 8} (compatible con embed_dim) |
| `n_blocks` | {2, 3, 4} |
| `out_ff_layers` | {"128-64", "256-128", "128-64-32"} |
| `dropout` | [0.0, 0.3] |
| `lr` | [1e-4, 5e-3] log |
| `wd` | [1e-6, 1e-3] log |
| `batch_size` | {1024, 2048, 4096} |

`max_epochs=10` en trials (early stopping patience=3). Reentrenamiento final: max_epochs=30, patience=7.

In [29]:
def make_objective(target):
    cols = CONT_FEATURES + CAT_FEATURES + [target]
    train_df = df_train[cols].copy()
    val_df   = df_val[cols].copy()

    n_neg = int((train_df[target] == 0).sum())
    n_pos = int((train_df[target] == 1).sum())
    # pytorch-tabular usa CrossEntropyLoss con 2 logits; weight=[w_neg, w_pos]
    class_weight = torch.tensor([1.0, n_neg / max(n_pos, 1)], device=device)

    def objective(trial):
        embed_dim    = trial.suggest_categorical('embed_dim', [16, 32, 64])
        valid_heads  = [h for h in [2, 4, 8] if embed_dim % h == 0]
        n_heads      = trial.suggest_categorical('n_heads', valid_heads)
        n_blocks     = trial.suggest_categorical('n_blocks', [2, 3, 4])
        out_ff       = trial.suggest_categorical('out_ff_layers', ['128-64', '256-128', '128-64-32'])
        dropout      = trial.suggest_float('dropout', 0.0, 0.3)
        lr           = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
        wd           = trial.suggest_float('wd', 1e-6, 1e-3, log=True)
        batch_size   = trial.suggest_categorical('batch_size', [1024, 2048, 4096])

        model = build_tabular_model(
            embed_dim=embed_dim, n_heads=n_heads, n_blocks=n_blocks,
            dropout=dropout, out_ff_layers=out_ff, lr=lr, wd=wd,
            batch_size=batch_size, target=target, max_epochs=10, patience=3,
        )
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weight)
        model.fit(train=train_df, validation=val_df, loss=loss_fn)
        auroc = get_auroc(model, val_df, target)
        del model
        torch.cuda.empty_cache()
        return auroc
    return objective

## 5. Optimización por target

Se ejecuta la búsqueda bayesiana para cada target con registro de tiempo y estadísticas de podado para documentar el presupuesto computacional.

In [20]:
best_params = {}
best_aurocs = {}
trial_times = {}

t_total_start = time.time()

for target in TARGETS:
    print(f"\n{'='*60}")
    print(f'Optimizando TARGET: {target} — {TARGET_NAMES[target]}')
    print(f"{'='*60}")

    t_start = time.time()
    study = optuna.create_study(
        direction='maximize',
        study_name=f'tabtrans_{target}',
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )
    study.optimize(make_objective(target), n_trials=N_TRIALS, show_progress_bar=True)
    elapsed = time.time() - t_start
    trial_times[target] = elapsed

    best_params[target] = study.best_params
    best_aurocs[target] = study.best_value
    print(f'\n{target} — Mejor AUROC: {study.best_value:.4f}')
    print(f'   Tiempo: {elapsed:.1f}s ({elapsed/60:.1f} min)')
    print(f'   Mejores parámetros: {study.best_params}')

t_total = time.time() - t_total_start
print(f"\nTotal Optuna: {t_total:.1f}s ({t_total/3600:.2f} h)")


Optimizando TARGET: L1 — Ingreso hospitalario


  0%|          | 0/30 [00:00<?, ?it/s]

2026-06-11 00:54:45,848 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off
Seed set to 42
2026-06-11 00:54:45,883 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-06-11 00:54:46,107 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for classification task
2026-06-11 00:54:46,865 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: TabTransformerModel
2026-06-11 00:54:46,989 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-06-11 00:54:47,037 - {pytorch_tabular.tabular_model:677} - INFO - Training Started
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`Trainer.fit


L1 — Mejor AUROC: 0.8240
   Tiempo: 4727.2s (78.8 min)
   Mejores parámetros: {'embed_dim': 16, 'n_heads': 4, 'n_blocks': 2, 'out_ff_layers': '256-128', 'dropout': 0.0396504831471077, 'lr': 0.0005683679223272175, 'wd': 9.912750032875996e-05, 'batch_size': 1024}

Optimizando TARGET: L2 — Resultado crítico


  0%|          | 0/30 [00:00<?, ?it/s]

2026-06-11 02:13:33,031 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off
Seed set to 42
2026-06-11 02:13:33,059 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-06-11 02:13:33,289 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for classification task
2026-06-11 02:13:34,043 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: TabTransformerModel
2026-06-11 02:13:34,173 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-06-11 02:13:34,226 - {pytorch_tabular.tabular_model:677} - INFO - Training Started
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`Trainer.fit


L2 — Mejor AUROC: 0.8806
   Tiempo: 3316.0s (55.3 min)
   Mejores parámetros: {'embed_dim': 16, 'n_heads': 4, 'n_blocks': 4, 'out_ff_layers': '128-64-32', 'dropout': 0.13082995007168405, 'lr': 0.0012901286770600152, 'wd': 2.1616135215579843e-05, 'batch_size': 4096}

Optimizando TARGET: L3 — Intervención crítica


  0%|          | 0/30 [00:00<?, ?it/s]

2026-06-11 03:08:48,981 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off
Seed set to 42
2026-06-11 03:08:49,011 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-06-11 03:08:49,248 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for classification task
2026-06-11 03:08:49,973 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: TabTransformerModel
2026-06-11 03:08:50,098 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-06-11 03:08:50,145 - {pytorch_tabular.tabular_model:677} - INFO - Training Started
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
2026-06-11 0


L3 — Mejor AUROC: 0.8917
   Tiempo: 2855.5s (47.6 min)
   Mejores parámetros: {'embed_dim': 64, 'n_heads': 8, 'n_blocks': 3, 'out_ff_layers': '128-64', 'dropout': 0.004031620659634916, 'lr': 0.0007116635150202098, 'wd': 4.456250818980063e-05, 'batch_size': 2048}

Total Optuna: 10898.6s (3.03 h)


## 6. Reentrenamiento final con mejores parámetros (max_epochs=30, patience=7)

Se reentrena con los hiperparámetros óptimos y el presupuesto de épocas completo para obtener el rendimiento real del modelo.

In [30]:
final_models  = {}
final_results = {}

for target in TARGETS:
    bp   = best_params[target]
    cols = CONT_FEATURES + CAT_FEATURES + [target]
    train_df = df_train[cols].copy()
    val_df   = df_val[cols].copy()

    n_neg = int((train_df[target] == 0).sum())
    n_pos = int((train_df[target] == 1).sum())
    class_weight = torch.tensor([1.0, n_neg / max(n_pos, 1)], device=device)
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weight)

    print(f"\n{'='*60}")
    print(f'REENTRENANDO: {target}  (max_epochs=30, patience=7, pos_weight={class_weight[1].item():.2f})')
    print(f"{'='*60}")

    model = build_tabular_model(
        embed_dim    = bp['embed_dim'],
        n_heads      = bp['n_heads'],
        n_blocks     = bp['n_blocks'],
        dropout      = bp['dropout'],
        out_ff_layers= bp['out_ff_layers'],
        lr           = bp['lr'],
        wd           = bp['wd'],
        batch_size   = bp['batch_size'],
        target       = target,
        max_epochs   = 30,
        patience     = 7,
        checkpoints  = 'valid_loss',
        load_best    = True,
    )
    model.fit(train=train_df, validation=val_df, loss=loss_fn)

    pred     = model.predict(val_df)
    y_pred   = pred[get_prob_col(pred)].values
    y_true   = val_df[target].values

    auroc = roc_auc_score(y_true, y_pred)
    auprc = average_precision_score(y_true, y_pred)
    brier = brier_score_loss(y_true, y_pred)
    print(f'  AUROC: {auroc:.4f} | AUPRC: {auprc:.4f} | Brier: {brier:.4f}')

    final_results[target] = {'AUROC': auroc, 'AUPRC': auprc, 'Brier': brier, 'Prev_val': float(y_true.mean())}
    final_models[target]  = model

print('\nReentrenamiento completado.')

2026-06-11 08:40:07,373 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off
Seed set to 42
2026-06-11 08:40:07,403 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders



REENTRENANDO: L1  (max_epochs=30, patience=7, pos_weight=1.59)


2026-06-11 08:40:07,628 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for classification task
2026-06-11 08:40:08,359 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: TabTransformerModel
2026-06-11 08:40:08,481 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-06-11 08:40:08,540 - {pytorch_tabular.tabular_model:677} - INFO - Training Started
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
2026-06-11 08:44:27,083 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-06-11 08:44:27,085 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model
2026-06-11 08:44:29,438 - {pytorch

  AUROC: 0.8243 | AUPRC: 0.7384 | Brier: 0.1727

REENTRENANDO: L2  (max_epochs=30, patience=7, pos_weight=63.79)


Seed set to 42
2026-06-11 08:44:29,466 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-06-11 08:44:29,688 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for classification task
2026-06-11 08:44:30,456 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: TabTransformerModel
2026-06-11 08:44:30,606 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-06-11 08:44:30,656 - {pytorch_tabular.tabular_model:677} - INFO - Training Started
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
2026-06-11 08:46:57,748 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-06-11 08:46:

  AUROC: 0.8754 | AUPRC: 0.1151 | Brier: 0.1503

REENTRENANDO: L3  (max_epochs=30, patience=7, pos_weight=177.07)


2026-06-11 08:47:00,598 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for classification task
2026-06-11 08:47:01,339 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: TabTransformerModel
2026-06-11 08:47:01,475 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-06-11 08:47:01,522 - {pytorch_tabular.tabular_model:677} - INFO - Training Started
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
2026-06-11 08:49:08,417 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-06-11 08:49:08,419 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model


  AUROC: 0.8874 | AUPRC: 0.0866 | Brier: 0.1336

Reentrenamiento completado.


## 7. Resumen de resultados Optuna

Métricas del modelo TabTransformer optimizado en validación. Las comparativas entre arquitecturas (LogReg, CatBoost, LSTM, TabTransformer, NAM) se realizan en el notebook de evaluación con el test set.

In [31]:
rows = []
for target in TARGETS:
    o = final_results[target]
    rows.append({
        'Target':   target,
        'Nombre':   TARGET_NAMES[target],
        'AUROC':    o['AUROC'],
        'AUPRC':    o['AUPRC'],
        'Brier':    o['Brier'],
        'Prev_val': o['Prev_val'],
    })
df_summary = pd.DataFrame(rows).set_index('Target')
print('=== TabTransformer + Optuna — Resultados en validación ===')
print(df_summary.round(4).to_string())

=== TabTransformer + Optuna — Resultados en validación ===
                      Nombre   AUROC   AUPRC   Brier  Prev_val
Target                                                        
L1      Ingreso hospitalario  0.8243  0.7384  0.1727    0.3831
L2         Resultado crítico  0.8754  0.1151  0.1503    0.0147
L3      Intervención crítica  0.8874  0.0866  0.1336    0.0061


## 8. Guardado de artefactos

Se persisten los pesos del modelo (`.pt`), los hiperparámetros óptimos, la configuración y los preprocesadores en `models/tabtransformer/<timestamp>/` para su uso en la fase de evaluación.

In [32]:
TIMESTAMP  = datetime.now().strftime('%Y%m%d_%H%M%S')
MODELS_DIR = Path(f'../../models/tabtransformer/{TIMESTAMP}')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# pytorch-tabular guarda modelo + configs completos (recargable sin código extra)
for target, model in final_models.items():
    model.save_model(str(MODELS_DIR / f'tabtransformer_{target}'))

with open(MODELS_DIR / 'best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)
with open(MODELS_DIR / 'final_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

model_config_save = {
    'cont_features':        CONT_FEATURES,
    'cat_features':         CAT_FEATURES,
    'chiefcomplaint_top_n': CHIEFCOMPLAINT_TOP_N,
    'top_cc':               sorted(top_cc),
    'best_params':          best_params,
}
with open(MODELS_DIR / 'model_config.json', 'w') as f:
    json.dump(model_config_save, f, indent=2, ensure_ascii=False)

# Imputer para inferencia
with open(MODELS_DIR / 'preprocessors.pkl', 'wb') as f:
    pickle.dump({'imputer': imputer}, f)

print(f'Artefactos guardados en {MODELS_DIR}')
print('Para recargar: TabularModel.load_from_checkpoint(str(MODELS_DIR / "tabtransformer_L1"))')

`weights_only` was not set, defaulting to `False`.
`weights_only` was not set, defaulting to `False`.
`weights_only` was not set, defaulting to `False`.


Artefactos guardados en ..\..\models\tabtransformer\20260611_085045
Para recargar: TabularModel.load_from_checkpoint(str(MODELS_DIR / "tabtransformer_L1"))
